# Week 1 · Math essentials for ML

This is the theory block (90 minutes a day), next to the Python work. It covers only the math the ML and DL weeks actually use. Each idea is explained in words, worked through by hand, then **checked with NumPy**. The drills at the end of each section are the kind of thing the exam asks you to do on paper. Pace: sections 1–2 Mon, 3–4 Tue, 5–6 Wed, 6–7 Thu, 7 Fri, 8–9 Sat.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **P(A)**: The probability of event A, a number from 0 to 1.
- **P(A | B)**: The probability of A **given that** B happened. Read the bar as "given".
- **random variable**: A number whose value depends on chance, e.g. the result of a die roll.
- **E[X]**: Expected value: the long-run average of X.
- **Var(X)**: Variance: how spread out X is. Its square root is the standard deviation σ.
- **likelihood**: How probable the **observed data** is, for a given choice of model parameters.
- **gradient ∇**: The vector of partial derivatives. It points in the direction where a function **increases** fastest.
- **loss**: A number that measures how wrong a model is. Training means making it small.

## 1. Probability and Bayes' rule

**Conditional probability:** P(A | B) = P(A and B) / P(B). In words: among the cases where B happens, what fraction also have A?

**Bayes' rule** flips the condition around:  **P(A | B) = P(B | A) · P(A) / P(B)**. It's useful because P(B | A) is often easy to know, while P(A | B) is the thing you actually want.

**Worked example.** 1% of people have a disease. The test catches 99% of sick people (P(+ | sick) = 0.99), and wrongly says + for 5% of healthy people (P(+ | healthy) = 0.05). You test +. What is the chance you're sick?

| Step | Calculation | Value |
|---|---|---|
| P(+ and sick) | 0.99 × 0.01 | 0.0099 |
| P(+ and healthy) | 0.05 × 0.99 | 0.0495 |
| P(+) = sum of the two | 0.0099 + 0.0495 | 0.0594 |
| P(sick \| +) | 0.0099 / 0.0594 | **≈ 0.167** |

Only about 17%, because healthy people are so much more common that their false positives outnumber the true positives. Check it by simulating a million people:

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n = 1_000_000
sick = rng.random(n) < 0.01                              # True for about 1% of people
positive = np.where(sick, rng.random(n) < 0.99,           # sick: + with prob 0.99
                          rng.random(n) < 0.05)           # healthy: + with prob 0.05
print(round(sick[positive].mean(), 3))                    # → close to 0.167

This is exactly how a **naive Bayes classifier** works (Week 2): P(class | features) ∝ P(features | class) · P(class).

**Drill (on paper).** 20% of emails are spam. The word "free" appears in 60% of spam and in 5% of normal emails. An email contains "free". What is P(spam | free)?

<details><summary>Worked answer</summary>

P(free and spam) = 0.6 × 0.2 = 0.12. P(free and normal) = 0.05 × 0.8 = 0.04. P(free) = 0.16. So P(spam | free) = 0.12 / 0.16 = **0.75**.

</details>

## 2. Expectation and variance

**Expected value** E[X] = Σ x · P(x), the probability-weighted average. **Variance** Var(X) = E[(X − E[X])²], the average squared distance from the mean. An easier formula to compute with: **Var(X) = E[X²] − (E[X])²**.

**Fair die:** E[X] = (1+2+3+4+5+6)/6 = 3.5. E[X²] = (1+4+9+16+25+36)/6 = 91/6 ≈ 15.17. Var = 15.17 − 3.5² = 15.17 − 12.25 ≈ **2.92**.

In [ ]:
faces = np.arange(1, 7)
p = np.ones(6) / 6
EX = (faces * p).sum()
EX2 = (faces**2 * p).sum()
print(EX, round(EX2 - EX**2, 3))       # → 3.5 2.917

**Two rules you'll use all the time:** E[aX + b] = a·E[X] + b, and **Var(aX + b) = a²·Var(X)**. Adding a constant doesn't change the spread, but scaling by a multiplies the variance by a².

**Drill (on paper).** P(X=0) = 0.5, P(X=1) = 0.3, P(X=2) = 0.2. Find E[X] and Var(X). Then find Var(3X + 1).

<details><summary>Worked answer</summary>

E[X] = 0 + 0.3 + 0.4 = 0.7. E[X²] = 0 + 0.3 + 0.8 = 1.1. Var = 1.1 − 0.49 = **0.61**. Var(3X + 1) = 9 × 0.61 = **5.49**.

</details>

## 3. The Gaussian (normal) distribution

The bell curve, with mean μ (the center) and standard deviation σ (the width). About **68%** of values fall within 1σ of the mean, **95%** within 2σ, and **99.7%** within 3σ.

**Standardizing:** z = (x − μ) / σ turns any normal into one with mean 0 and std 1. It's the same formula as the NumPy feature standardization.

In [ ]:
x = rng.normal(loc=170, scale=10, size=100_000)     # heights: μ = 170, σ = 10
inside_1sd = np.abs(x - 170) < 10
print(round(inside_1sd.mean(), 2))                   # → close to 0.68

**More than one variable:** a multivariate Gaussian has a **mean vector** and a **covariance matrix** Σ. The diagonal of Σ holds each variable's variance, and the off-diagonal entries hold the covariances (positive means the two tend to go up together).

In [ ]:
h = rng.normal(170, 10, 5000)
w = 0.9 * h + rng.normal(0, 5, 5000) - 90            # weight goes up with height
print(np.cov(h, w).round(0))                         # 2×2 covariance matrix
# → [[ 98.  88.]
#    [ 88. 104.]]   diagonal = the variances (in theory 100 and 106);
#                   off-diagonal = the covariance (in theory 90). Positive: they rise together

## 4. Maximum likelihood (MLE)

**The idea:** choose the parameters that make the data you actually observed **as probable as possible**.

**Coin example.** You flip a coin 10 times and get 7 heads. If the probability of heads is p, the likelihood of exactly this data is L(p) = p⁷ (1 − p)³. Which p makes it biggest?

**Step 1, take the log.** The log turns products into sums and has its maximum in the same place: log L = 7 log p + 3 log(1 − p).

**Step 2, set the derivative to 0.** 7/p − 3/(1 − p) = 0  →  7(1 − p) = 3p  →  7 = 10p  →  **p = 0.7**. That's just the fraction of heads, which is what intuition says too.

In [ ]:
p = np.linspace(0.01, 0.99, 99)
log_L = 7 * np.log(p) + 3 * np.log(1 - p)
print(round(p[np.argmax(log_L)], 2))     # → 0.7

**For a Gaussian**, the MLE of the mean μ is the **sample average**, and the MLE of σ² is the average squared deviation.

**Why this matters for the whole course: every common loss function is an MLE in disguise.**

| Loss you minimize | Is MLE for |
|---|---|
| mean squared error (regression) | a Gaussian noise model: y = prediction + Gaussian noise |
| binary cross-entropy (logistic regression) | a Bernoulli model of the label |
| cross-entropy (softmax classifier) | a categorical model of the label |

**Drill (on paper).** You see 3 heads in 12 flips. What is the MLE of p? Show the derivative step.

<details><summary>Worked answer</summary>

log L = 3 log p + 9 log(1 − p). The derivative is 3/p − 9/(1 − p) = 0 → 3 − 3p = 9p → **p = 0.25**.

</details>

## 5. MAP: adding a prior belief

MLE uses only the data. **MAP** (maximum a posteriori) also uses a **prior**, which is what you believed before seeing the data. By Bayes' rule: posterior ∝ likelihood × prior, and MAP picks the peak of the posterior.

For a coin, the standard prior is a **Beta(a, b)** distribution. The MAP estimate then has a simple formula: **p̂ = (heads + a − 1) / (n + a + b − 2)**.

With a Beta(2, 2) prior (a mild belief that the coin is fair) and 7 heads in 10: p̂ = (7 + 1)/(10 + 2) = 8/12 ≈ **0.667**. That's pulled from 0.7 toward 0.5. With 700 heads in 1000, MAP gives 701/1002 ≈ 0.6996, almost the same as MLE. **The more data you have, the less the prior matters.**

**The link to ML:** putting a Gaussian prior on a model's weights and doing MAP is the same as adding the penalty λ‖w‖² to the loss. That's **ridge regression** and **weight decay** (Weeks 2 and 6).

**Drill (on paper).** Same data as the last drill (3 heads in 12), with a Beta(3, 3) prior. What is the MAP estimate, and which way did it move compared with the MLE?

<details><summary>Worked answer</summary>

(3 + 2)/(12 + 4) = 5/16 = **0.3125**. It moved from 0.25 toward 0.5, the center of the prior.

</details>

## 6. Linear algebra for ML

**Dot product:** u · v = Σ uᵢvᵢ. It's large when the two vectors point the same way, 0 when they're perpendicular, and negative when they point opposite ways. It's the basic operation of every model: prediction = **w · x** + b.

**Matrix × vector:** each output entry is one **row** of the matrix dotted with the vector. A (m, n) matrix times a (n,) vector gives a (m,) vector.

In [ ]:
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])      # (3, 2)
x = np.array([1, 10])       # (2,)
print(A @ x)                # → [21 43 65]    row 0: 1·1 + 2·10 = 21, and so on
print(np.linalg.norm(np.array([3, 4])))   # → 5.0    length of a vector: √(3² + 4²)

**Eigenvectors.** For a square matrix A, an eigenvector v is a direction that A only **stretches** and doesn't rotate: **A v = λ v**. The stretch factor λ is the eigenvalue. To find the eigenvalues by hand, solve **det(A − λI) = 0**.

**By hand, A = [[2, 1], [1, 2]]:** det = (2 − λ)² − 1 = 0 → 2 − λ = ±1 → **λ = 1 or 3**. For λ = 3: (A − 3I)v = 0 gives −v₁ + v₂ = 0, so v ∝ (1, 1). For λ = 1: v ∝ (1, −1).

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
vals, vecs = np.linalg.eigh(A)     # eigh: for symmetric matrices, eigenvalues come out sorted
print(vals)                        # → [1. 3.]
v = vecs[:, 1]                     # eigenvector for λ = 3 (a COLUMN of vecs)
print(A @ v, 3 * v)                # A v equals 3 v: both ≈ [2.12 2.12]

**Why you care:** PCA (Week 4) is the eigenvectors of the data's **covariance matrix**. The eigenvector with the biggest eigenvalue is the direction the data varies the most.

**SVD**, for any matrix: A = U Σ Vᵀ. For now just know that it exists, that it always works (even when A isn't square), and that it's how PCA is computed in practice: `np.linalg.svd`.

**Drill (on paper).** Find the eigenvalues of [[4, 0], [0, 1]] and of [[3, 1], [1, 3]]. Then compute [[1, 2], [3, 4]] @ [[0, 1], [1, 0]].

<details><summary>Worked answer</summary>

[[4,0],[0,1]]: **4 and 1** (a diagonal matrix's eigenvalues are its diagonal). [[3,1],[1,3]]: (3 − λ)² = 1 → **λ = 4, 2**. The product: row 0 is [1·0 + 2·1, 1·1 + 2·0] = [2, 1], row 1 is [3·0 + 4·1, 3·1 + 4·0] = [4, 3], so **[[2, 1], [4, 3]]**. Multiplying by this matrix swaps the columns.

</details>

## 7. Gradients and gradient descent

Training a model means **finding the weights that make the loss smallest**. The gradient ∇L points uphill, so we take small steps **downhill**:  **w ← w − η ∇L(w)**, where η (eta) is the **learning rate**.

**One dimension, by hand.** L(w) = (w − 3)², so L'(w) = 2(w − 3). Start at w = 0 with η = 0.1:

| Step | gradient 2(w − 3) | new w = w − 0.1 × gradient |
|---|---|---|
| 0 → 1 | 2(0 − 3) = −6 | 0 + 0.6 = **0.6** |
| 1 → 2 | 2(0.6 − 3) = −4.8 | 0.6 + 0.48 = **1.08** |
| 2 → 3 | 2(1.08 − 3) = −3.84 | 1.08 + 0.384 = **1.464** |

In [ ]:
w = 0.0
for step in range(50):
    grad = 2 * (w - 3)
    w = w - 0.1 * grad
print(round(w, 4))       # → 3.0    it reaches the minimum

**The least-squares gradient, the one result to remember.** For a linear model with predictions Xw, the loss is L(w) = ‖Xw − y‖² = Σᵢ (xᵢ·w − yᵢ)². Its gradient is **∇L = 2 Xᵀ(Xw − y)**.

**Where it comes from:** for one sample, (x·w − y)² has derivative 2(x·w − y)·x by the chain rule (the inside derivative of x·w with respect to w is x). Summing over the samples stacks those x's: Σᵢ 2(xᵢ·w − yᵢ) xᵢ = 2Xᵀ(Xw − y).

Setting it to zero gives the **normal equation XᵀX w = Xᵀy**, the closed-form solution of linear regression (Week 2). Check both claims numerically:

In [ ]:
X = rng.normal(size=(20, 3))
y = rng.normal(size=20)
w = rng.normal(size=3)
loss = lambda w: np.sum((X @ w - y) ** 2)

analytic = 2 * X.T @ (X @ w - y)
eps = 1e-6
numeric = np.array([(loss(w + eps * e) - loss(w - eps * e)) / (2 * eps) for e in np.eye(3)])
print(np.allclose(analytic, numeric))                       # → True   the formula is right

w_best = np.linalg.solve(X.T @ X, X.T @ y)                  # solve the normal equation
print(np.allclose(2 * X.T @ (X @ w_best - y), 0))           # → True   gradient is 0 there

**Drill (on paper).** f(w) = w², starting at w = 1, learning rate 0.25. Do two gradient-descent steps by hand.

<details><summary>Worked answer</summary>

f'(w) = 2w. Step 1: w = 1 − 0.25 × 2 = **0.5**. Step 2: w = 0.5 − 0.25 × 1 = **0.25**. Each step halves w.

</details>

## 8. Overfitting, bias–variance, and train/validation/test

A model that is too **simple** misses the pattern (**underfitting**, high bias). A model that is too **flexible** memorizes the noise (**overfitting**, high variance). See it with polynomials of different degree fitted to noisy data:

In [ ]:
x_train = np.linspace(0, 1, 15);  y_train = np.sin(2 * np.pi * x_train) + rng.normal(0, 0.2, 15)
x_test  = np.linspace(0, 1, 200); y_test  = np.sin(2 * np.pi * x_test)  + rng.normal(0, 0.2, 200)
for degree in [1, 4, 12]:
    coefs = np.polyfit(x_train, y_train, degree)
    mse = lambda x, y: np.mean((np.polyval(coefs, x) - y) ** 2)
    print(f"degree {degree:2d}: train MSE {mse(x_train, y_train):.3f}   test MSE {mse(x_test, y_test):.3f}")
# → degree  1: train MSE 0.258   test MSE 0.253    both high: underfit
#   degree  4: train MSE 0.027   test MSE 0.053    both low: good
#   degree 12: train MSE 0.003   test MSE 0.117    train tiny, test 2× worse: overfit

**The bias–variance decomposition:** expected test error = **bias²** + **variance** + **irreducible noise**. More flexibility lowers bias but raises variance, and the best model is in the middle.

| Split | Used for | Rule |
|---|---|---|
| train | fitting the weights | the model sees it |
| validation | choosing settings: degree, λ, learning rate | used many times, while developing |
| test | the final, honest score | touch it **once**, at the very end |

**Data leakage** means information from the validation or test data sneaking into training (e.g. standardizing with the mean of *all* the data before splitting). Your KAMP copied-days finding was a real example of this.

## 9. Entropy, cross-entropy, KL divergence

**Entropy** H(p) = −Σ pᵢ log pᵢ measures uncertainty. A fair coin has H = 1 bit (using log base 2). A coin that always lands heads has H = 0, since there's no uncertainty. Decision trees (Week 3) split data to reduce entropy.

**Cross-entropy** H(p, q) = −Σ pᵢ log qᵢ compares the true distribution p with the model's prediction q. In classification, p is **one-hot** (1 for the true class, 0 for the rest), so it simplifies to just **−log q_true**. That's the classification loss in every neural network.

In [ ]:
def cross_entropy(true_class, q):
    return -np.log(q[true_class])

print(round(cross_entropy(2, np.array([0.1, 0.2, 0.7])), 3))   # → 0.357   confident and right: small loss
print(round(cross_entropy(2, np.array([0.6, 0.3, 0.1])), 3))   # → 2.303   confident and wrong: big loss

**KL divergence** KL(p ‖ q) = H(p, q) − H(p) ≥ 0 measures how far q is from p. It's 0 only when q = p. It appears in VAEs (Week 7).

In [ ]:
p = np.array([0.5, 0.5]); q = np.array([0.9, 0.1])
H = lambda p: -np.sum(p * np.log2(p))
print(H(p), round(H(q), 3))                       # → 1.0 0.469
KL = np.sum(p * np.log(p / q))
print(round(KL, 3))                               # → 0.511

**Drill (on paper).** True class 0, prediction q = [0.5, 0.25, 0.25]. What is the cross-entropy loss (natural log)? What is the entropy of q in bits?

<details><summary>Worked answer</summary>

−ln 0.5 = **0.693**. H(q) = −(0.5·log₂0.5 + 2 × 0.25·log₂0.25) = 0.5 × 1 + 0.5 × 2 = **1.5 bits**.

</details>

---
The Sunday quiz (**`05_sunday_quiz.ipynb`**) has 5 math questions like these drills. Log any drill you got wrong.